In [0]:
import joblib
import pandas as pd
import random
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# --------------------------------------------------
# Load Model
# --------------------------------------------------
model = joblib.load(
    "/Workspace/Users/Team_Files/rf_transaction_return_numeric.pkl"
)

# --------------------------------------------------
# Load Feature List
# --------------------------------------------------
feature_cols = joblib.load(
    "/Workspace/Users/Team_Files/rf_transaction_return_numeric_features.pkl"
)

# --------------------------------------------------
# Read Feature Engineered CSV
# --------------------------------------------------
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Workspace/Users/Team_Files/ecommerce_orders_rf_features_numeric_sample.csv")
)

# simulate incoming data
fraction = random.randint(10, 18)/100
df = df.sample(fraction=fraction, seed=None)
print(f"Sampling {fraction:.0%} of the dataset")

# Convert to Pandas
pdf = df.toPandas()

# --------------------------------------------------
# Prediction
# --------------------------------------------------
X = pdf[feature_cols]

preds = model.predict(X)

# Convert numeric prediction to readable labels
pdf["return_prediction"] = pd.Series(preds).map({
    0: "Not Returned",
    1: "Returned"
})

# --------------------------------------------------
# Save to Delta Table
# --------------------------------------------------
preds_df = spark.createDataFrame(pdf)

preds_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.ecommerce_return_classification")

print("Return Prediction Completed")

print(pdf["return_prediction"].value_counts())

/databricks/python/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Sampling 15% of the dataset
Return Prediction Completed
return_prediction
Not Returned    1138
Returned         355
Name: count, dtype: int64


In [0]:
display(
    pdf[["returned", "return_prediction"]].head(20)
)

returned,return_prediction
1,Returned
0,Not Returned
0,Returned
0,Not Returned
0,Not Returned
0,Not Returned
0,Not Returned
0,Not Returned
0,Not Returned
0,Not Returned
